# Lab 4: Memory & Persistence (Applied Version)
In this notebook, we'll evolve our **Smart Content Generation System** to support multi-turn collaboration.

Rather than auto-evaluating, we want the system to remember previous states using a thread checkpointer. This allows the user to chat with the generation node to iterate on draft scripts interactively. The agent reads the thread history to understand requested edits.

We use the following concepts:
- **MemorySaver**: Storing checkpoint states indexed by `thread_id`.
- **Chat History Reducers**: Automatically appending user requests and agent replies using `operator.add`.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### 1. Define State and Node

In [2]:
from typing import TypedDict, List, Annotated
import operator
from mock_llm import get_llm
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class InteractiveContentState(TypedDict):
    topic: str
    draft: str
    # Tracks full user-agent conversation history
    messages: Annotated[List[dict], operator.add]
    user_feedback: str

llm = get_llm(model="gpt-4o-mini", temperature=0.7)

def writer_agent_node(state: InteractiveContentState):
    print("--- Node: Executing Writer Agent ---")
    
    # Format chat history from dict list into langchain message objects
    formatted_history = []
    for m in state.get("messages", []):
        formatted_history.append((m["role"], m["content"]))
        
    # Set up prompt structure incorporating conversation history
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a collaborative copywriting bot.
Your goal is to write and refine a short blog paragraph based on user instructions.
Current draft: {draft}
If no draft exists, generate one from the topic. If a draft exists, update it based on user feedback.
Output your response in two parts: your conversational message and the updated draft, separated by '---DRAFT---'.
Example:
Here is the updated draft incorporating your edits.\n---DRAFT---\nThis is the draft text..."""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "Topic: {topic}\nFeedback: {user_feedback}")
    ])
    
    chain = prompt | llm
    response = chain.invoke({
        "draft": state.get("draft", ""),
        "topic": state.get("topic", ""),
        "user_feedback": state.get("user_feedback", "Start"),
        "chat_history": formatted_history
    })
    
    output = response.content.strip()
    if "---DRAFT---" in output:
        conversation_msg, new_draft = output.split("---DRAFT---")
        conversation_msg = conversation_msg.strip()
        new_draft = new_draft.strip()
    else:
        conversation_msg = output
        new_draft = state.get("draft", "")
        
    # Append messages to chat history via list addition
    feedback_msg = state.get("user_feedback", "Generate initial draft")
    return {
        "draft": new_draft,
        "messages": [
            {"role": "user", "content": f"Topic: {state['topic']}. Feedback: {feedback_msg}"},
            {"role": "assistant", "content": conversation_msg}
        ]
    }

--- OpenAI API connection failed (Error code: 429 - {'error': {'message': 'You exceeded your c...). Falling back to Mock LLM ---


### 2. Build and Compile with Memory

In [3]:
builder = StateGraph(InteractiveContentState)
builder.add_node("writer", writer_agent_node)
builder.add_edge(START, "writer")
builder.add_edge("writer", END)

# Set up thread checkpointer memory
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

### 3. Run and Collaborate (Turn 1 & 2)
We set a unique `thread_id` and test interactive drafting.

In [4]:
config = {"configurable": { "thread_id": "collab-thread-1" }}

# Turn 1: Generate initial draft
print("--- Conversation Turn 1: Initial Generation ---")
state = graph.invoke({
    "topic": "The importance of cybersecurity for small businesses",
    "draft": "",
    "messages": [],
    "user_feedback": "Make it brief and punchy"
}, config)

print("\nAgent Message:", state["messages"][-1]["content"])
print("Current Draft:\n", state["draft"])

# Turn 2: Request revision on the same thread
print("\n--- Conversation Turn 2: Requesting Edits ---")
state = graph.invoke({
    "topic": "The importance of cybersecurity for small businesses",
    "draft": "", # Checkpoint recovers the draft, so we can pass empty
    "messages": [],
    "user_feedback": "Can you add a warning about phishing scams?"
}, config)

print("\nAgent Message:", state["messages"][-1]["content"])
print("Updated Draft:\n", state["draft"])

--- Conversation Turn 1: Initial Generation ---
--- Node: Executing Writer Agent ---

Agent Message: Cybersecurity is vital for small businesses to protect their client data. Cyberattacks, particularly phishing scams, frequently target smaller companies because of weak defenses. Implementing protocols safeguards trust and assets.
Current Draft:
 

--- Conversation Turn 2: Requesting Edits ---
--- Node: Executing Writer Agent ---

Agent Message: Cybersecurity is vital for small businesses to protect their client data. Cyberattacks, particularly phishing scams, frequently target smaller companies because of weak defenses. Implementing protocols safeguards trust and assets.
Updated Draft:
 
